# 02 — Target & feature distributions, outliers (team 07)

**Owner:** Lasya · **Branch:** `team-07-srilasyakamma`

**Questions to answer**
1. **Target distribution** — histogram of `TARGET` raw and on a log scale. Skewed? Zeros? Negative
   values? Two populations (like price-by-cabin in D2)?
2. **Outliers** — NumPy/IQR outlier counts for the target and the main numeric features. Are the
   outliers errors, or real (huge hub routes)? Real outliers stay — but they change the loss we pick.
3. **Key numeric features** — distributions of the numeric columns you expect to matter
   (`describe()`, histograms). Which need a log transform or scaling?
4. Does the target's spread differ a lot across groups (big vs small airports, route types)?

**Write the "why", not just the plot.** Every plot or number needs a sentence after it:
*"…therefore we should ___ when we model."* Those sentences go in the table at the bottom.

> **NDA reminder.** This is real American Airlines data. Never commit it, never paste rows or
> screenshots of it anywhere outside the team, and **clear outputs before every commit**
> (Kernel → Restart & Clear Output) so no data ends up in the notebook file.

In [ ]:
%run ./00_load_and_preprocess.ipynb
# gives you: pax, airports, TARGET, REPO_ROOT, RESULTS_DIR


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style="whitegrid")

# Confirm that the preprocessing notebook loaded the data
print("Passenger dataset shape:", pax.shape)
print("Airport dataset shape:", airports.shape)
print("Target variable:", TARGET)

# Check that the target exists
assert TARGET in pax.columns, f"{TARGET} not found in pax"

# Identify numeric columns
# Unnamed: 0 is the CSV's row counter (01), so it is not a real feature
numeric_cols = [c for c in pax.select_dtypes(include=np.number).columns if c != "Unnamed: 0"]

print("Number of numeric columns:", len(numeric_cols))

## 1. Target distribution

In [ ]:
# Basic descriptive statistics
target_stats = pax[TARGET].describe()

display(target_stats)

# Check missing, zero, and negative values
print("Missing values:", pax[TARGET].isna().sum())
print("Zero values:", (pax[TARGET] == 0).sum())
print("Negative values:", (pax[TARGET] < 0).sum())

# Calculate skewness
print("Target skewness:", pax[TARGET].skew())



In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(
    data=pax,
    x=TARGET,
    bins=50,
    kde=True
)

plt.title("Distribution of TARGET (Raw)")
plt.xlabel(TARGET)
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
# log1p requires values greater than or equal to -1.
# Passenger counts should normally be nonnegative.
if pax[TARGET].dropna().min() >= 0:

    target_log = np.log1p(pax[TARGET].dropna())

    plt.figure(figsize=(10, 5))

    sns.histplot(
        target_log,
        bins=50,
        kde=True
    )

    plt.title("Distribution of TARGET (Log1p)")
    plt.xlabel(f"log(1 + {TARGET})")
    plt.ylabel("Frequency")

    plt.tight_layout()
    plt.show()

else:
    print("Negative target values found. Investigate before applying log1p.")

OD_PAX has about 1.5 million values and none of them are missing, zero, or negative. The smallest value is a fraction of a passenger, so these counts are estimates rather than whole passengers. The raw distribution is very right-skewed (skew 5.3). The median route-month has about 503 passengers, but the mean is about 2,765, because a small number of huge routes pull the average up. After taking log(1 + passengers), the distribution is close to symmetric (skew -0.12), with the mean and median both about 6.2. It looks like one wide population, not two separate groups.

What this means for the model: we should train on log(1 + OD_PAX) so the biggest routes don't dominate the fit, and then turn the predictions back into passengers when we measure error.

## 2. Outliers (IQR)

In [ ]:
def calculate_iqr_outliers(series):
    """
    Calculate IQR outlier statistics for a numeric pandas Series.
    Missing values are excluded.
    """

    values = series.dropna().to_numpy()

    if len(values) == 0:
        return {
            "Q1": np.nan,
            "Q3": np.nan,
            "IQR": np.nan,
            "Lower Bound": np.nan,
            "Upper Bound": np.nan,
            "Outlier Count": 0,
            "Outlier Percentage": np.nan
        }

    q1, q3 = np.percentile(values, [25, 75])

    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outliers = (
        (values < lower_bound) |
        (values > upper_bound)
    )

    return {
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "Outlier Count": int(outliers.sum()),
        "Outlier Percentage": outliers.mean() * 100
    }


# Calculate outliers for all numeric columns
outlier_results = {}

for col in numeric_cols:
    outlier_results[col] = calculate_iqr_outliers(pax[col])

outlier_df = pd.DataFrame(outlier_results).T

display(
    outlier_df.round(2)
)

In [ ]:
plt.figure(figsize=(10, 5))

sns.boxplot(
    x=pax[TARGET].dropna()
)

plt.title(f"Boxplot of {TARGET}")
plt.xlabel(TARGET)

plt.tight_layout()
plt.show()

Using the IQR rule, about 13.6% of route-months are outliers for passengers and 12.7% for revenue. All of them are on the high side, because the lower cutoff is below zero. These are not data errors. About two thirds of the passenger outliers come from the top 10% of routes, which are the big hub-to-hub markets.

What this means for the model: we should keep the outliers, since they carry most of AA's passengers. Instead of removing them, we should handle them with the log target and judge the model with a size-weighted error like WAPE.

## 3. Numeric feature distributions

In [ ]:
# Summary statistics for numeric features
numeric_summary = pax[numeric_cols].describe().T

# Add skewness
numeric_summary["skewness"] = pax[numeric_cols].skew()

# Add missing value counts
numeric_summary["missing_values"] = pax[numeric_cols].isna().sum()

display(numeric_summary.round(2))

In [ ]:
# Plot numeric features in groups of 4
features_to_plot = [
    col for col in numeric_cols
    if col != TARGET
]

for start in range(0, len(features_to_plot), 4):

    cols = features_to_plot[start:start + 4]

    pax[cols].hist(
        figsize=(14, 9),
        bins=30,
        layout=(2, 2),
        edgecolor="black"
    )

    plt.suptitle(
        "Numeric Feature Distributions",
        fontsize=15
    )

    plt.tight_layout()
    plt.show()

In [ ]:
skewness = pax[numeric_cols].skew().sort_values(
    ascending=False
)

display(skewness.to_frame("Skewness"))

# Identify strongly skewed positive numeric features
log_candidates = []

for col in numeric_cols:

    values = pax[col].dropna()

    if len(values) == 0:
        continue

    if values.min() >= 0 and abs(values.skew()) > 1:
        log_candidates.append(col)

print("Potential log transformation candidates:")
print(log_candidates)

Besides the target, the only numeric columns are DEP_MONTH, DEP_YEAR, and OD_REV_USD (Unnamed: 0 is a row counter, so it is left out). Month and year are spread out evenly (skew about 0.1 and 0.0), so they don't need a transform, but month should be encoded as a cycle so December sits next to January. OD_REV_USD is even more skewed than passengers (skew 12.1) and would need a log transform.

What this means for the model: OD_REV_USD for the same month should not be a feature at all, because notebooks 03 and 04 show it is only known after the month happens. Only past revenue per passenger can be used.

## 4. Spread by group

In [ ]:
# Show column names and data types
column_info = pd.DataFrame({
    "Column": pax.columns,
    "Data Type": pax.dtypes.astype(str).values,
    "Unique Values": pax.nunique(dropna=True).values
})

display(column_info)

In [ ]:
# Origin airport: compares big and small airports (question 4)
GROUP_COL = "ORG_AIRPRT"

if GROUP_COL not in pax.columns:
    raise ValueError(
        f"{GROUP_COL} does not exist. Choose a column from pax.columns."
    )

# Create a temporary analysis dataframe
group_data = pax[[TARGET, GROUP_COL]].dropna().copy()

# Keep the most common 10 groups for readability
top_groups = (
    group_data[GROUP_COL]
    .value_counts()
    .head(10)
    .index
)

group_data = group_data[
    group_data[GROUP_COL].isin(top_groups)
].copy()

# Log-transform target for plotting
if (group_data[TARGET] < 0).any():
    raise ValueError("Negative target values need investigation.")

group_data["TARGET_LOG"] = np.log1p(
    group_data[TARGET]
)

plt.figure(figsize=(13, 6))

sns.boxplot(
    data=group_data,
    x=GROUP_COL,
    y="TARGET_LOG"
)

plt.title(f"Log Target Distribution by {GROUP_COL}")
plt.xlabel(GROUP_COL)
plt.ylabel(f"log(1 + {TARGET})")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Looking at the 10 origin airports with the most route-months, the typical route size changes a lot by airport. The median route-month ranges from about 700 to about 4,700 passengers depending on the origin. The spread inside each airport is also wide: the middle half of its routes covers about a 6 to 13 times range in passengers.

What this means for the model: which airports a route connects, and the route's own history, both matter a lot. The model should get origin and destination as features along with route-level history, and errors should be weighted by route size.

## Findings → modelling consequence (Lasya)

This table is what the team's feature set (05) is built from. Every row needs a **consequence**,
not just an observation — "*therefore we should ___ when we model.*"

| # | Observation (with the number) | Modelling consequence |
|---|---|---|
| 1 | OD_PAX has no zeros or negatives. Skew is 5.3 raw but -0.12 after log1p. Median about 503 vs mean about 2,765 | Train on log(1 + OD_PAX) and report errors back in passengers |
| 2 | The IQR rule flags 13.6% of route-months as high outliers, and about two thirds of them are on the top 10% of routes | Keep them, since they are real hub routes. Use a size-weighted metric (WAPE) and size-weighted training instead of dropping them |
| 3 | OD_REV_USD is even more skewed (12.1) and is same-month revenue | Do not use it as a feature (leakage, see 03 and 04). Use only past revenue per passenger |
| 4 | Median route size by origin airport ranges from about 700 to 4,700 passengers | Give the model origin and destination airports and each route's own history |